# ADMET · Distribution assessment for rapamycin and its rapalogs

Third in the series (potency → **A**bsorption → **D**istribution). Same molecules
throughout: rapamycin (sirolimus), everolimus, temsirolimus, ridaforolimus, zotarolimus.
Claims are grounded in primary literature (PubMed), cited.

## Purpose

**Distribution** asks: once in the blood, *where does the drug go* — how much is free vs.
bound, how widely does it partition into tissues, and does it reach the brain? The
predictable endpoints are **fu** (fraction unbound / plasma protein binding), **Vss**
(volume of distribution), **B:P** (blood-to-plasma ratio), and **BBB** penetration.

## Conclusion
1. Again, ML models were all trained on small molecules, making rapalogs out of applicable domain for using these models.
2. Used *measured* data for **fu, Vss, and B:P**, and *in silico* model for **BBB penetration** as a qualitative guide, not quantitative.
    1. Dominant distribution mode = FKBP12-driven RBC + tissue sequestration (*measured* B:P ≈ 36, whole-blood monitoring; wide tissue distribution) — un-modelable so measured data are decisive.
    2. BBB penetration *in silico* predicted: CNS MPO and BOILED-Egg both call the rapalogs **non-CNS**, matching measured poor brain penetration.
    3. Vss, fu, B:P are decided by measured data. ML VDss/PPB models are out of domain for rapalogs.

## Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import Draw, Descriptors
from chembl_webresource_client.new_client import new_client

# Notebook is stripped before push; each step's result is saved as a PNG here.
RESULTS = Path("../result_rapamycin_ADMET_Distribution"); RESULTS.mkdir(exist_ok=True)
def save_fig(fig, name): fig.savefig(RESULTS / name, dpi=150, bbox_inches="tight")
def save_img(obj, name):
    p = RESULTS / name
    obj.save(p) if hasattr(obj, "save") else p.write_bytes(obj.data)
def save_df_png(df, name, title=None):
    fig, ax = plt.subplots(figsize=(min(2 + len(df.columns) * 1.9, 16), 0.7 + 0.45 * len(df))); ax.axis("off")
    if title: ax.set_title(title, fontsize=10, loc="left")
    t = ax.table(cellText=df.astype(str).values, colLabels=list(df.columns), loc="center", cellLoc="center")
    t.auto_set_font_size(False); t.set_fontsize(8); t.scale(1, 1.4); t.auto_set_column_width(range(len(df.columns)))
    fig.savefig(RESULTS / name, dpi=150, bbox_inches="tight"); plt.close(fig)

mol_client = new_client.molecule
RAPALOGS = {"Sirolimus (rapamycin)": "CHEMBL413", "Everolimus": "CHEMBL1908360",
            "Temsirolimus": "CHEMBL1201182", "Ridaforolimus": "CHEMBL2103839",
            "Zotarolimus": "CHEMBL219410"}
smiles = {n: mol_client.filter(molecule_chembl_id=c).only(["molecule_structures"])[0]
          ["molecule_structures"]["canonical_smiles"] for n, c in RAPALOGS.items()}
mols = {n: Chem.MolFromSmiles(s) for n, s in smiles.items()}
print("Loaded", len(mols), "structures.")

## Step 1 — Honest caveat first

**The 900 Da problem, again.** Rapamycin and its rapalogs are 900–1030 Da macrocycles,
outside the "small drug-like" chemical space on which distribution QSARs (fu, Vss, BBB)
were trained and validated. ***De novo* *in silico* distribution analysis from structure alone
is unreliable for these molecules** — the same applicability-domain penalty we hit for
Absorption. So the decisive numbers come from **measured** pharmacokinetics (Step 3), and
*in silico* is used only to rationalize and rank.

**What makes their distribution genuinely un-modelable:** sirolimus is **~95% sequestered
inside red blood cells** (blood-to-plasma ratio **B:P ≈ 36**) via high-affinity binding to
its target protein **FKBP12**, which is present in erythrocytes and tissues throughout the
body (MacDonald et al., 2000; Mahalati & Kahan, 2001). No structure-only model predicts
target-protein-driven RBC/tissue sequestration — it is the dominant feature of rapalog
distribution and it is invisible to generic QSAR.

**Is there any macrocycle-trained ML to lean on?** Emerging, not established: recent work
applies ML with **conformal prediction** to macrocycles up to ~1200 Da (Fagerholm et al.,
2022) — but for *absorption*, and even for absorption it mainly returns wide *uncertainty intervals* for such
compounds. A validated **distribution** (Vss/PPB) model for non-peptide macrocycles does
not yet exist.

In [ ]:
tbl = pd.DataFrame({"MW (Da)": {n: round(Descriptors.MolWt(m)) for n, m in mols.items()},
                    "cLogP": {n: round(Descriptors.MolLogP(m), 1) for n, m in mols.items()},
                    "TPSA": {n: round(Descriptors.TPSA(m)) for n, m in mols.items()}})
print(tbl.to_string())
save_df_png(tbl.reset_index().rename(columns={"index": "drug"}),
            "step1_size_lipophilicity.png", "Step 1 - size & lipophilicity (why D is hard)")
grid = Draw.MolsToGridImage(list(mols.values()), legends=list(mols.keys()), molsPerRow=3, subImgSize=(300, 230))
save_img(grid, "step1_structures.png")
grid

## Step 2 — A *in silico*/measured distribution stack

**With rapalogs in this notebook:** Ran **BBB via CNS MPO and BOILED-Egg** to qualitatively predict BBB penetration. Used measured physicochemical panel (fu, Vss, and B:P are taken from *measured* data; Step 3).

Distribution modelling splits into **three tiers**, and which you can trust for a 900 Da
macrocycle differs sharply:

| Endpoint | Model / tier | Trustworthy for rapalogs? | Reference / data |
|---|---|---|---|
| **Vss** (volume of distribution) | **Mechanistic** tissue-composition (Rodgers–Rowland; Poulin) → Kp per tissue | *If fed measured* logP, pKa, fu, B:P — degrades gracefully | Rodgers et al., 2005; Poulin et al., 2023; Assmus et al., 2017 |
| **Vss** | **ML/QSAR** (RF on descriptors) | ❌ out of domain (trained on small molecules) | Lombardo et al., 2020 |
| **fu / plasma protein binding** | ML QSAR | ❌ out of domain; very-high-binding hard | (small-molecule ML) |
| **B:P** (RBC partitioning) | — | ❌ driven by FKBP12 binding — essentially unpredictable | measured only |
| **BBB penetration** | **BOILED-Egg** (WLOGP–TPSA yolk); **CNS MPO** desirability | ✅ *works qualitatively* (big + polar → non-CNS) | Daina & Zoete, 2016; Wager et al., 2010, 2016 |

> **Dominant distribution mode for rapalogs: RBC + tissue sequestration via FKBP12** Unmodelable..

## Step 3 — Measured distribution data

Key findings from clinical/PK literature (Citations in Step 6):

- **Sirolimus:** **B:P ≈ 36** (extensive partitioning into blood cells), whole-blood
  apparent **Vss ≈ 1.7 L/kg**, long terminal t½ ≈ 62 h, and it distributes widely in
  tissues — dose is monitored in whole blood, not plasma, precisely because of RBC
  sequestration (MacDonald et al., 2000; Mahalati & Kahan, 2001).
- **Everolimus:** blood-cell partitioning and plasma protein binding characterised across
  species; **brain penetration is poor** (dose-dependent, with a longer brain t½)(O'Reilly et al., 2009).
- **Temsirolimus / ridaforolimus / zotarolimus:** share the class's high lipophilicity and
  tissue distribution; zotarolimus was engineered lipophilic for *local* stent retention.

In [ ]:
measured = pd.DataFrame([
    {"drug": "Sirolimus", "B:P ratio": "~36 (RBC-sequestered)", "Vss": "~1.7 L/kg (whole-blood)",
     "plasma_protein_binding": "~92%", "brain_penetration": "low", "ref": "MacDonald 2000; Mahalati 2001"},
    {"drug": "Everolimus", "B:P ratio": "high (< sirolimus)", "Vss": "large / wide tissue",
     "plasma_protein_binding": "~74%", "brain_penetration": "poor (dose-dependent)", "ref": "O'Reilly 2009"},
    {"drug": "Temsirolimus", "B:P ratio": "high", "Vss": "wide tissue",
     "plasma_protein_binding": "high", "brain_penetration": "low", "ref": "class property"},
    {"drug": "Ridaforolimus", "B:P ratio": "high", "Vss": "wide tissue",
     "plasma_protein_binding": "high", "brain_penetration": "low", "ref": "class property"},
    {"drug": "Zotarolimus", "B:P ratio": "high", "Vss": "wide tissue (lipophilic by design)",
     "plasma_protein_binding": "high", "brain_penetration": "low", "ref": "class property"},
])
save_df_png(measured, "step3_measured_distribution_data.png", "Step 3 - measured (wet-lab) distribution data")
measured

### Summary of measured findings 

Distribution of this class is dominated by 

1. **FKBP12-driven RBC and tissue sequestration** (high B:P, whole-blood monitoring, wide tissue distribution)
2. **low CNS penetration**. The whole-blood-referenced Vss (~1.7 L/kg) *understates* the
true extent of tissue distribution because so much drug is hidden in blood cells.

## Step 4 — Minimally defensible *in silico* step: BBB penetration 

BBB penetration is the one distribution endpoint *in silico* predicts well from
structure, because it is governed by size/polarity/lipophilicity. Here, I compute, 
1. **CNS MPO** desirability score (Wager et al., 2010; six
physicochemical desirabilities, 0–6, higher = more CNS-like), and
2. **BOILED-Egg** yolk test.

In [ ]:
# CNS MPO desirability (Wager et al. 2010): six 0-1 desirability functions summed to 0-6.
def _ramp_down(x, good, bad):  # 1 up to `good`, linear to 0 at `bad`
    if x <= good: return 1.0
    if x >= bad: return 0.0
    return (bad - x) / (bad - good)

def _tpsa_hump(x):  # 0 <=20, up to 1 at 40, flat to 90, down to 0 at 120
    if x <= 20 or x >= 120: return 0.0
    if x < 40: return (x - 20) / 20.0
    if x <= 90: return 1.0
    return (120 - x) / 30.0

def cns_mpo(m):
    clogp = Descriptors.MolLogP(m); mw = Descriptors.MolWt(m)
    tpsa = Descriptors.TPSA(m); hbd = Descriptors.NumHDonors(m)
    # rapalogs are non-basic (no ionizable base) -> cLogD ~ cLogP; pKa term = 1
    d = {"cLogP": _ramp_down(clogp, 3, 5), "cLogD": _ramp_down(clogp, 2, 4),
         "MW": _ramp_down(mw, 360, 500), "TPSA": _tpsa_hump(tpsa),
         "HBD": _ramp_down(hbd, 0.5, 3.5), "pKa(basic)": 1.0}
    return sum(d.values()), d

rows = []
for n, m in mols.items():
    score, d = cns_mpo(m)
    rows.append({"drug": n.split(" ")[0], "MW": round(Descriptors.MolWt(m)),
                 "TPSA": round(Descriptors.TPSA(m)), "cLogP": round(Descriptors.MolLogP(m), 1),
                 "CNS_MPO(/6)": round(score, 2),
                 "CNS-likely (>=4)": "yes" if score >= 4 else "NO",
                 "BBB (BOILED-Egg yolk)": "no (TPSA>>79)" if Descriptors.TPSA(m) > 79 else "maybe"})
bbb = pd.DataFrame(rows)
save_df_png(bbb, "step4_bbb_cns_mpo.png", "Step 4 - BBB: CNS MPO desirability + BOILED-Egg yolk")
bbb

In [ ]:
# Visualize CNS MPO vs the CNS-likely threshold
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.bar(bbb["drug"], bbb["CNS_MPO(/6)"], color="slateblue")
ax.axhline(4, color="crimson", ls="--", label="CNS-likely threshold (>=4)")
ax.set(ylabel="CNS MPO desirability (0-6)", ylim=(0, 6),
       title="Rapalogs CNS MPO desirability score below the CNS threshold")
ax.legend(fontsize=8); plt.tight_layout()
save_fig(fig, "step4_cns_mpo_scores.png")
plt.show()
print("All rapalogs: CNS MPO ~", round(bbb['CNS_MPO(/6)'].min(), 1), "-", round(bbb['CNS_MPO(/6)'].max(), 1),
      "(<< 4) and TPSA >> 79 -> predicted NON-CNS, consistent with measured poor brain penetration.")

### Summary of *in silico* findings

- ***In silico* BBB penetration prediction matches measured data** Every rapalog scores **CNS MPO ~1–2 (far below CNS threshold at 4)** and sits far outside the BOILED-Egg yolk (TPSA ≈ 195–242 ≫ 79). Prediction: **non-CNS** — which *matches* the measured poor brain penetration (O'Reilly et al., 2009).

## Step 5 — Summary and caveats

**Findings.**
1. **Dominant distribution mode = FKBP12-driven RBC + tissue sequestration** (*measured* B:P ≈ 36;
   whole-blood monitoring; wide tissue distribution) — un-modelable so measured data are decisive.
2. **BBB penetration *in silico* predicted:** CNS MPO and BOILED-Egg both call the
   rapalogs **non-CNS**, matching measured poor brain penetration.
3. **Vss, fu, B:P are decided by measured data.** ML VDss/PPB models are out of domain for rapalogs.

**Caveats.**
- **900 Da applicability domain:** *de novo*, structure-only distribution prediction is unreliable
  for these macrocycles (except the size/polarity-driven BBB).
- **Emerging macrocycle ML:** conformal-prediction methods (Fagerholm et al., 2022) are the right
  direction — they *quantify* the large uncertainty for bRo5 compounds rather than hide it — but a
  validated distribution model for rapalog-like macrocycles does not yet exist.
- **Target protein(FKBP12)-driven RBC/tissue sequestration is unmodelable yet.**. Whole-blood Vss (~1.7 L/kg) understates tissue distribution because of RBC sequestration.

**Bottom line.** *In silico* distribution for the rapalogs is a **BBB screen that qualitatively works** plus a set
of endpoints (Vss/fu/B:P) where **measured data is decisive** and target-driven sequestration is
beyond structure-only models.

## Step 6 — Citations

Bibliographic metadata retrieved from **PubMed**.

**Drug distribution pharmacokinetics**

1. MacDonald, A.; Scarola, J.; Burke, J. T.; Zimmerman, J. J. Clinical Pharmacokinetics and
   Therapeutic Drug Monitoring of Sirolimus. *Clin. Ther.* **2000**, *22* (Suppl. B), B101–B121.
   https://doi.org/10.1016/S0149-2918(00)89027-X.
2. Mahalati, K.; Kahan, B. D. Clinical Pharmacokinetics of Sirolimus. *Clin. Pharmacokinet.*
   **2001**, *40* (8), 573–585. https://doi.org/10.2165/00003088-200140080-00002.
3. O'Reilly, T.; McSheehy, P. M. J.; Kawai, R.; Kretz, O.; McMahon, L.; Brueggen, J.;
   Bruelisauer, A.; Gschwind, H.-P.; Allegrini, P. R.; Lane, H. A. Comparative Pharmacokinetics
   of RAD001 (Everolimus) in Normal and Tumor-Bearing Rodents. *Cancer Chemother. Pharmacol.*
   **2010**, *65* (4), 625–639. https://doi.org/10.1007/s00280-009-1068-8.

***In silico* distribution methods**

4. Rodgers, T.; Leahy, D.; Rowland, M. Physiologically Based Pharmacokinetic Modeling 1:
   Predicting the Tissue Distribution of Moderate-to-Strong Bases. *J. Pharm. Sci.* **2005**,
   *94* (6), 1259–1276. https://doi.org/10.1002/jps.20322.
5. Poulin, P.; Nicolas, J.-M.; Bouzom, F. A New Version of the Tissue Composition-Based Model
   for Improving the Mechanism-Based Prediction of Volume of Distribution at Steady-State for
   Neutral Drugs. *J. Pharm. Sci.* **2024**, *113* (1), 118–130.
   https://doi.org/10.1016/j.xphs.2023.08.018.
6. Assmus, F.; Houston, J. B.; Galetin, A. Incorporation of Lysosomal Sequestration in the
   Mechanistic Model for Prediction of Tissue Distribution of Basic Drugs. *Eur. J. Pharm. Sci.*
   **2017**, *109*, 419–430. https://doi.org/10.1016/j.ejps.2017.08.014.
7. Lombardo, F.; Bentzien, J.; Berellini, G.; Muegge, I. In Silico Models of Human PK
   Parameters. Prediction of Volume of Distribution Using an Extensive Data Set and a Reduced
   Number of Parameters. *J. Pharm. Sci.* **2021**, *110* (1), 500–509.
   https://doi.org/10.1016/j.xphs.2020.08.023.
8. Wager, T. T.; Hou, X.; Verhoest, P. R.; Villalobos, A. Moving beyond Rules: The Development
   of a Central Nervous System Multiparameter Optimization (CNS MPO) Approach To Enable
   Alignment of Druglike Properties. *ACS Chem. Neurosci.* **2010**, *1* (6), 435–449.
   https://doi.org/10.1021/cn100008c.
9. Wager, T. T.; Hou, X.; Verhoest, P. R.; Villalobos, A. Central Nervous System Multiparameter
   Optimization Desirability: Application in Drug Discovery. *ACS Chem. Neurosci.* **2016**,
   *7* (6), 767–775. https://doi.org/10.1021/acschemneuro.6b00029.
10. Daina, A.; Zoete, V. A BOILED-Egg To Predict Gastrointestinal Absorption and Brain
    Penetration of Small Molecules. *ChemMedChem* **2016**, *11* (11), 1117–1121.
    https://doi.org/10.1002/cmdc.201600182.

**Machine learning for macrocycles (emerging model)**

11. Fagerholm, U.; Hellberg, S.; Alvarsson, J.; Spjuth, O. In Silico Predictions of the
    Gastrointestinal Uptake of Macrocycles in Man Using Conformal Prediction Methodology.
    *J. Pharm. Sci.* **2022**, *111* (9), 2614–2619. https://doi.org/10.1016/j.xphs.2022.05.010.

*Attribution: bibliographic data retrieved from PubMed. Plasma-protein-binding percentages and
some class properties reflect approved product labels / review values.*